In [10]:
import os
import cv2
import numpy as np
from PIL import Image, ImageChops, ImageEnhance
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split



In [11]:
# ==========================================
# STEP 1: Error Level Analysis (ELA) Processing
# ==========================================
def compute_ela(image_path, quality=90):
    """
    Computes the Error Level Analysis of an image to spot tampered regions.
    """
    temp_filename = '/content/certificate.png'

    # Open original image and save with target compression quality
    original = Image.open(image_path).convert('RGB')
    original.save(temp_filename, 'JPEG', quality=quality)

    # Open temporary compressed image
    compressed = Image.open(temp_filename)

    # Calculate absolute pixel difference between original and compressed image
    ela_image = ImageChops.difference(original, compressed)

    # Extrapolate the differences so they are easily visible
    extrema = ela_image.getextrema()
    max_diff = max([ex[1] for ex in extrema])
    if max_diff == 0:
        max_diff = 1
    scale = 255.0 / max_diff

    ela_image = ImageEnhance.Brightness(ela_image).enhance(scale)

    # Clean up temp file
    if os.path.exists(temp_filename):
        os.remove(temp_filename)

    return ela_image

def prepare_image(image_path, target_size=(128, 128)):
    """
    Converts image to ELA format and resizes it for CNN entry.
    """
    ela_img = compute_ela(image_path)
    ela_img = ela_img.resize(target_size)
    return np.array(ela_img) / 255.0  # Normalize pixel values



In [12]:
# ==========================================
# STEP 2: Dummy Dataset Generator for Training
# ==========================================
def load_dataset_placeholder():
    """
    Creates dummy matrix shapes representing your structured real vs fake data.
    Replace path strings below with your structured image folders.
    """
    # X will hold image vectors, Y will hold binary labels (0 = Genuine, 1 = Fake)
    X, Y = [], []

    # Note: Populate these folders with sample certificate images to test
    # for img_name in os.listdir('dataset/genuine'):
    #     X.append(prepare_image(os.path.join('dataset/genuine', img_name)))
    #     Y.append(0)

    # Mock generation loops for demonstration purposes
    for _ in range(50):
        X.append(np.random.rand(128, 128, 3))
        Y.append(0)  # Genuine class
    for _ in range(50):
        X.append(np.random.rand(128, 128, 3))
        Y.append(1)  # Fake/Tampered class

    return np.array(X), np.array(Y)





In [13]:
# ==========================================
# STEP 3: CNN Model Definition & Construction
# ==========================================
def build_cnn_model(input_shape=(128, 128, 3)):
    """
    Compiles a Deep Learning CNN architecture optimized for visual fraud metrics.
    """
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')  # Binary Output: Close to 1 means Fake
    ])

    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

In [14]:
# ==========================================
# STEP 4: Main Pipeline Logic
# ==========================================
if __name__ == "__main__":
    print("[-] Initializing dataset generation...")
    X, Y = load_dataset_placeholder()

    # Split into Training and Validation sets
    X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

    print("[-] Constructing Neural Network architecture...")
    detector_model = build_cnn_model()

    print("[-] Commencing ML Model Training Loop...")
    detector_model.fit(X_train, Y_train, epochs=5, batch_size=8, validation_data=(X_val, Y_val))

    # Save the trained model parameters
    detector_model.save('certificate_detector.h5')
    print("[+] Model pipeline complete. Saved configuration to 'certificate_detector.h5'.")



[-] Initializing dataset generation...
[-] Constructing Neural Network architecture...
[-] Commencing ML Model Training Loop...
Epoch 1/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 291ms/step - accuracy: 0.5375 - loss: 0.7937 - val_accuracy: 0.6000 - val_loss: 0.6744
Epoch 2/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 266ms/step - accuracy: 0.5125 - loss: 0.7121 - val_accuracy: 0.4000 - val_loss: 0.7068
Epoch 3/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 372ms/step - accuracy: 0.5250 - loss: 0.7053 - val_accuracy: 0.4000 - val_loss: 0.6940
Epoch 4/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 270ms/step - accuracy: 0.5250 - loss: 0.6930 - val_accuracy: 0.4000 - val_loss: 0.6941
Epoch 5/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 258ms/step - accuracy: 0.5250 - loss: 0.6931 - val_accuracy: 0.4000 - val_loss: 0.6959


[+] Model pipeline complete. Saved configuration to 'certificate_detector.h5'.


In [20]:
# ==========================================
    # STEP 5: Testing Single Uploaded Certificate
    # ==========================================
def verify_certificate(file_path):
      """
      Infers legitimacy of a specific single target document.
      """
      processed_input = prepare_image(file_path)
      processed_input = np.expand_dims(processed_input, axis=0) # Add batch axis

      # Load weights and predict
      model = tf.keras.models.load_model('certificate_detector.h5')
      prediction = model.predict(processed_input)[0][0]

      print(f"\n[Verification Metrics Matrix Report for: {file_path}]")
      print(f"Tamper Confidence Score Index: {prediction * 100:.2f}%")

      if prediction > 0.5:
          print("❌ WARNING: The document presents structural anomalies. Classified as FAKE.")
      else:
          print("✅ SUCCESS: Document compression signature matches patterns. Classified as GENUINE.")
          verify_certificate('test_certificate.png')
          # To run validation verification, uncomment line below after adding an image: